In [7]:
import sys
from pathlib import Path

import gymnasium as gym
import numpy as np
import torch
from pystk2_gymnasium import AgentSpec
from tqdm.auto import trange

from dagger_rlpd_stk import (
    ContinuousObservationNormalizer,
    DAggerConfig,
    ReplayBuffer,
    RLPDConfig,
    STKRLPDWrapper,
    compute_observation_statistics,
    evaluate,
    normalize_replay_buffer,
    run_dagger,
    run_rlpd,
)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pystk_project").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT / "pystk_project" / "src"))

# RLPD on SuperTuxKart

This notebook adapts SuperTuxKart to the flat, normalized spaces expected by RLPD.

## Environment adapter

The wrapper concatenates the continuous state features with a one-hot encoding of every discrete feature. The same vector can be consumed by RLPD and DAgger. It exposes acceleration and steering in `[-1, 1]`; acceleration is mapped back to STK's native `[0, 1]` interval before each step. Fixed statistics from the Agent 3 dataset standardize the continuous prefix for both algorithms, while the one-hot suffix remains unchanged.

In [8]:
ENV_NAME = "supertuxkart/flattened_continuous_actions-v0"
PLAYER_NAME = "Jean"
TRACK_NAME = "abyss"  # Use this same track for evaluation and training.


def make_stk_env(render_mode=None):
    env = gym.make(
        ENV_NAME,
        render_mode=render_mode,
        track=TRACK_NAME,
        agent=AgentSpec(name=PLAYER_NAME, rank_start=0),
    )
    return STKRLPDWrapper(env)

In [9]:
check_env = make_stk_env()
state, _ = check_env.reset(seed=0)
print("state shape:", state.shape)
print("observation space:", check_env.observation_space)
print("normalized action space:", check_env.action_space)
assert state.shape == check_env.observation_space.shape
assert np.all(check_env.action_space.low == -1.0)
assert np.all(check_env.action_space.high == 1.0)
check_env.close()

..:: Antarctica Rendering Engine 2.0 ::..
state shape: (160,)
observation space: Box(-inf, inf, (160,), float32)
normalized action space: Box(-1.0, 1.0, (2,), float32)


## Evaluate an example policy

The imported example policies consume the full observation dictionary and produce native STK action dictionaries, including braking, drifting, nitro, rescue, and item use. Evaluate them on the native environment before adapting one into an offline-data policy. `policy_factory` should accept the environment and return a fresh policy with a `choose_action(observation)` method.

In [10]:
def evaluate_policy(
    policy_factory,
    episodes=5,
    max_steps=1_500,
    render_mode=None,
    seed=0,
):
    evaluation_env = gym.make(
        "supertuxkart/full-v0",
        render_mode=render_mode,
        track=TRACK_NAME,
        agent=AgentSpec(name="Policy evaluation", rank_start=0),
    )
    episode_results = []

    try:
        for episode in range(episodes):
            observation, _ = evaluation_env.reset(seed=seed + episode)
            policy = policy_factory(evaluation_env.unwrapped)
            episode_return = 0.0
            terminated = truncated = False
            info = {}

            for step in range(1, max_steps + 1):
                action = policy.choose_action(observation)
                observation, reward, terminated, truncated, info = evaluation_env.step(
                    action
                )
                episode_return += float(reward)
                if terminated or truncated:
                    break

            episode_results.append(
                {
                    "return": episode_return,
                    "length": step,
                    "finished": bool(terminated),
                    "distance": float(info.get("distance", 0.0)),
                    "position": int(info.get("position", 0)),
                }
            )
    finally:
        evaluation_env.close()

    completed_lengths = [
        result["length"] for result in episode_results if result["finished"]
    ]
    returns = [result["return"] for result in episode_results]

    summary = {
        "mean_return": float(np.mean(returns)),
        "return_std": float(np.std(returns)),
        "completion_rate": float(
            np.mean([result["finished"] for result in episode_results])
        ),
        "mean_completion_length": (
            float(np.mean(completed_lengths)) if completed_lengths else float("inf")
        ),
        "mean_distance": float(
            np.mean([result["distance"] for result in episode_results])
        ),
    }
    return summary, episode_results

In [11]:
from agents.team1.agent1 import Agent1
from agents.team3.agent3 import Agent3
from agents.team4.agent4 import Agent4
from agents.team5.agent5 import Agent5
from agents.team6.agent6 import Agent6
from agents.team7.agent7 import Agent7

policy_classes = {
    1: Agent1,
    3: Agent3,
    4: Agent4,
    5: Agent5,
    6: Agent6,
    7: Agent7,
}

policy_evaluations = {}
for agent_number, policy_class in policy_classes.items():
    summary, episode_results = evaluate_policy(
        lambda policy_env, policy_class=policy_class: policy_class(policy_env),
        episodes=3,
    )
    policy_evaluations[agent_number] = {
        "summary": summary,
        "episodes": episode_results,
    }
    print(f"Agent {agent_number}: {summary}")

..:: Antarctica Rendering Engine 2.0 ::..
Agent 1: {'mean_return': 312.49105902777774, 'return_std': 143.9347889664768, 'completion_rate': 1.0, 'mean_completion_length': 635.6666666666666, 'mean_distance': 1262.7994791666667}
..:: Antarctica Rendering Engine 2.0 ::..
Agent 3: {'mean_return': 395.5729899088544, 'return_std': 174.91662268021938, 'completion_rate': 1.0, 'mean_completion_length': 580.3333333333334, 'mean_distance': 1262.7298990885417}
..:: Antarctica Rendering Engine 2.0 ::..
Agent 4: {'mean_return': 374.57869737413193, 'return_std': 158.61537765952897, 'completion_rate': 1.0, 'mean_completion_length': 594.6666666666666, 'mean_distance': 1262.6758626302083}
..:: Antarctica Rendering Engine 2.0 ::..
Agent 5: {'mean_return': 281.01175537109356, 'return_std': 75.76414284175158, 'completion_rate': 1.0, 'mean_completion_length': 596.0, 'mean_distance': 1262.7842203776042}
..:: Antarctica Rendering Engine 2.0 ::..
Agent 6: {'mean_return': -0.7929097493497111, 'return_std': 1.990

We find that agent 3 is the best one in general.

## Offline data

Assign the demonstration policy to `good_policy` when it is available. It only needs a `select_action(state)` method. Use a separate environment for collection so its state does not interfere with training or evaluation.

In [12]:
from agents.team3.agent3 import Agent3


def agent3_action(action):
    acceleration = float(np.asarray(action["acceleration"]).item())
    steering = float(np.asarray(action["steer"]).item())
    return np.array(
        [2.0 * acceleration - 1.0, steering],
        dtype=np.float32,
    )


def collect_agent3_offline_buffer(num_transitions, seed=0):
    offline_env = make_stk_env()
    state_dim = offline_env.observation_space.shape[0]
    action_dim = offline_env.action_space.shape[0]
    continuous_dim = offline_env.continuous_observation_dim
    offline_buffer = ReplayBuffer(state_dim, action_dim, num_transitions)

    try:
        state, _ = offline_env.reset(seed=seed)
        native_env = offline_env.unwrapped
        policy = Agent3(native_env)

        for _ in trange(num_transitions, desc="Collecting Agent 3 data"):
            native_observation = native_env.get_observation(
                native_env.kart_ix,
                False,
            )
            action = agent3_action(policy.choose_action(native_observation))
            next_state, reward, terminated, truncated, _ = offline_env.step(action)

            offline_buffer.add(
                torch.as_tensor(state, dtype=torch.float32),
                torch.as_tensor(action, dtype=torch.float32),
                torch.as_tensor(next_state, dtype=torch.float32),
                reward,
                terminated,
                truncated,
            )

            if terminated or truncated:
                state, _ = offline_env.reset()
                policy = Agent3(native_env)
            else:
                state = next_state
    finally:
        offline_env.close()

    return offline_buffer, continuous_dim


EXPERT_DATASET_SIZE = 100_000
offline_buffer, continuous_observation_dim = collect_agent3_offline_buffer(
    num_transitions=EXPERT_DATASET_SIZE,
    seed=0,
)
observation_mean, observation_std = compute_observation_statistics(
    offline_buffer,
    continuous_observation_dim,
)
normalize_replay_buffer(
    offline_buffer,
    observation_mean,
    observation_std,
)


def make_learning_env(render_mode=None):
    return ContinuousObservationNormalizer(
        make_stk_env(render_mode=render_mode),
        observation_mean.numpy(),
        observation_std.numpy(),
    )

..:: Antarctica Rendering Engine 2.0 ::..


## DAgger training

DAgger uses exactly the same continuous-plus-one-hot state vector as RLPD. Agent 3 is queried through the native STK observation because its hand-written controller predates the flattened learning interface. Its output is converted to the normalized action space used by the learning environment.

In [13]:
class Agent3DAggerExpert:
    def __init__(self, wrapped_env):
        self.native_env = wrapped_env.unwrapped
        self.policy = Agent3(self.native_env)

    def __call__(self, encoded_observation):
        # The encoded observation is what DAgger learns from. Agent 3 needs
        # the corresponding native dictionary observation to provide a label.
        native_observation = self.native_env.get_observation(
            self.native_env.kart_ix,
            False,
        )
        return agent3_action(self.policy.choose_action(native_observation))

    def reset(self):
        # Do not call Agent3.reset(): run_dagger has already reset the env.
        self.policy = Agent3(self.native_env)

In [14]:
dagger_config = DAggerConfig(
    n_iterations=50,
    beta_p=0.0,
    ridge_lambda=1e-3,
    results_dir=PROJECT_ROOT / "notebooks" / "dagger_results",
    seed=0,
)

dagger_env = make_learning_env()
dagger_evaluation_env = make_learning_env()
try:
    dagger_expert = Agent3DAggerExpert(dagger_env)
    dagger_policy, dagger_history = run_dagger(
        env=dagger_env,
        expert=dagger_expert,
        validate=lambda policy: evaluate(
            policy, dagger_evaluation_env, eval_episodes=5
        ),
        config=dagger_config,
        log_dir=PROJECT_ROOT / "notebooks" / "runs" / "stk_dagger",
    )
finally:
    dagger_env.close()
    dagger_evaluation_env.close()

..:: Antarctica Rendering Engine 2.0 ::..
..:: Antarctica Rendering Engine 2.0 ::..


DAgger:   0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
dagger_history

## RLPD training

The initial configuration is intentionally small enough for development. Increase the ensemble to 10 critics and the UTD ratio to 20 only after the full pipeline works.

In [ ]:
config = RLPDConfig(
    replay_buffer_size=EXPERT_DATASET_SIZE,
    hidden_dim=256,
    layer_normalization=True,
    batch_size=256,
    start_timesteps=0,
    max_timesteps=500_000,
    eval_episodes=5,
    eval_freq=2_000,
    n_ensemble_critics=3,
    n_subset_critics=2,
    utd_ratio=2,
    entropy=False,
    normalize_observations=False,
    seed=0,
)

In [ ]:
# The same fixed, normalized Agent 3 dataset supplies RLPD's offline data.
assert offline_buffer.size == config.replay_buffer_size

In [ ]:
training_env = make_learning_env()
evaluation_env = make_learning_env()

try:
    policy, evaluations = run_rlpd(
        env=training_env,
        evaluation_env=evaluation_env,
        offline_buffer=offline_buffer,
        config=config,
        save_results="stk_rlpd_evaluations.pt",
        log_dir="runs/stk_rlpd",
    )
finally:
    training_env.close()
    evaluation_env.close()

In [ ]:
def evaluate_rlpd_policy(policy, episodes=20, seed=10_000):
    env = make_learning_env()
    episode_results = []
    policy.actor.eval()

    try:
        for episode in range(episodes):
            state, _ = env.reset(seed=seed + episode)
            episode_return = 0.0
            terminated = truncated = False
            info = {}
            length = 0

            while not (terminated or truncated):
                action = policy.select_action(
                    state,
                    deterministic=True,
                )

                state, reward, terminated, truncated, info = env.step(action)
                episode_return += float(reward)
                length += 1

            episode_results.append(
                {
                    "return": episode_return,
                    "length": length,
                    "finished": bool(terminated),
                    "distance": float(info.get("distance", 0.0)),
                    "position": int(info.get("position", 0)),
                }
            )
    finally:
        env.close()

    completed_lengths = [
        result["length"] for result in episode_results if result["finished"]
    ]

    summary = {
        "mean_return": float(np.mean([result["return"] for result in episode_results])),
        "return_std": float(np.std([result["return"] for result in episode_results])),
        "completion_rate": float(
            np.mean([result["finished"] for result in episode_results])
        ),
        "mean_completion_length": (
            float(np.mean(completed_lengths)) if completed_lengths else float("inf")
        ),
        "mean_distance": float(
            np.mean([result["distance"] for result in episode_results])
        ),
    }

    return summary, episode_results

In [ ]:
rlpd_summary, rlpd_episode_results = evaluate_rlpd_policy(
    policy,
    episodes=100,
)

rlpd_summary

In [ ]:
summary, episode_results = evaluate_policy(Agent3, episodes=100)
print(summary)